# Meta Ads — Performance Notebook

Pull campaign data from your connected Meta Ads account and explore it with charts and tables.

**Setup:** Paste your `session_id` in Cell 1, then run all cells (`Runtime → Run all`).

In [ ]:
# ── Cell 1: Config ──────────────────────────────────────────────────────────
# Paste your session_id from the Ads Agents dashboard.
# Never share this — it gives full access to your ad account.
SESSION_ID = "paste-your-session-id-here"

BASE_URL = "https://ads-agents-production.up.railway.app/api"

# Date range for analysis (YYYY-MM-DD). Leave as None for all-time.
DATE_FROM = "2025-01-01"
DATE_TO   = None

# Platform filter: "meta", "google", or None for all
PLATFORM = "meta"

In [ ]:
# ── Cell 2: Install & imports ────────────────────────────────────────────────
%pip install -q requests pandas plotly

import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

HEADERS = {"X-Api-Key": SESSION_ID}

def get(path, **params):
    """GET /api/<path> with session auth. Raises on HTTP errors."""
    params = {k: v for k, v in params.items() if v is not None}
    r = requests.get(f"{BASE_URL}/{path}", headers=HEADERS, params=params)
    r.raise_for_status()
    return r.json()

print("Setup OK")

---
## Campaigns overview

In [ ]:
# ── Cell 3: All campaigns table ──────────────────────────────────────────────
data = get("campaigns", date_from=DATE_FROM, date_to=DATE_TO, platform=PLATFORM)
campaigns = data["campaigns"]
print(f"{data['total']} campaigns")

rows = []
for c in campaigns:
    m = c.get("metrics", {})
    rows.append({
        "campaign_id":  c["campaign_id"],
        "name":         c.get("name", ""),
        "status":       c.get("status", ""),
        "objective":    c.get("objective", ""),
        "spend":        m.get("total_spend", 0),
        "leads":        m.get("total_leads", 0),
        "cpl":          m.get("cpl", 0),
        "ctr%":         m.get("avg_ctr", 0),
        "impressions":  m.get("total_impressions", 0),
        "roas":         m.get("roas", 0),
    })

df_campaigns = pd.DataFrame(rows).sort_values("spend", ascending=False)
display(df_campaigns.style.format({
    "spend":       "${:.2f}",
    "cpl":         "${:.2f}",
    "ctr%":        "{:.2f}%",
    "impressions": "{:,.0f}",
    "roas":        "{:.2f}x",
}).background_gradient(subset=["spend", "leads"], cmap="Blues"))

In [ ]:
# ── Cell 4: Spend vs CPL scatter ─────────────────────────────────────────────
fig = px.scatter(
    df_campaigns[df_campaigns["spend"] > 0],
    x="spend", y="cpl",
    size="leads", color="status",
    hover_name="name",
    title="Spend vs CPL (bubble = leads)",
    labels={"spend": "Total Spend ($)", "cpl": "Cost Per Lead ($)"},
)
fig.show()

In [ ]:
# ── Cell 5: Top 10 campaigns by spend ────────────────────────────────────────
top10 = df_campaigns.head(10)
fig = px.bar(
    top10, x="spend", y="name", orientation="h",
    color="cpl", color_continuous_scale="RdYlGn_r",
    title="Top 10 Campaigns — Spend (color = CPL)",
    labels={"spend": "Spend ($)", "cpl": "CPL ($)"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

---
## Campaign deep-dive

Set `CAMPAIGN_ID` to any campaign from the table above.

In [ ]:
# ── Cell 6: Pick a campaign ───────────────────────────────────────────────────
CAMPAIGN_ID = df_campaigns.iloc[0]["campaign_id"]  # auto: top spender
print(f"Drilling into: {CAMPAIGN_ID}")

In [ ]:
# ── Cell 7: Daily spend trend ────────────────────────────────────────────────
detail = get(f"campaigns/{CAMPAIGN_ID}", date_from=DATE_FROM, date_to=DATE_TO)
m = detail.get("metrics", {})
print(f"Campaign: {detail.get('name')}")
print(f"  Spend: ${m.get('total_spend', 0):,.2f}  |  Leads: {m.get('total_leads', 0)}  |  CPL: ${m.get('cpl', 0):.2f}")
print(f"  CTR: {m.get('avg_ctr', 0):.2f}%  |  ROAS: {m.get('roas', 0):.2f}x  |  CPM: ${m.get('cpm', 0):.2f}")

df_daily = pd.DataFrame(detail.get("daily_insights", []))
if not df_daily.empty:
    df_daily["date"] = pd.to_datetime(df_daily["date"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_daily["date"], y=df_daily["spend"], name="Spend", fill="tozeroy"))
    if "leads" in df_daily.columns:
        fig.add_trace(go.Bar(x=df_daily["date"], y=df_daily["leads"], name="Leads", yaxis="y2", opacity=0.5))
    fig.update_layout(
        title=f"Daily Spend & Leads — {detail.get('name')}",
        yaxis={"title": "Spend ($)"},
        yaxis2={"title": "Leads", "overlaying": "y", "side": "right"},
        legend={"orientation": "h"},
    )
    fig.show()

In [ ]:
# ── Cell 8: Placements breakdown ─────────────────────────────────────────────
pl = get(f"campaigns/{CAMPAIGN_ID}/placements", date_from=DATE_FROM, date_to=DATE_TO)
df_pl = pd.DataFrame(pl.get("placements", []))

if not df_pl.empty:
    df_pl["label"] = df_pl["publisher"].str.title() + " / " + df_pl["position"].str.replace("_", " ").str.title()
    fig = px.bar(
        df_pl.sort_values("spend", ascending=False),
        x="label", y="spend", color="ctr",
        color_continuous_scale="Greens",
        title="Placements — Spend (color = CTR)",
        labels={"label": "Placement", "spend": "Spend ($)", "ctr": "CTR"},
    )
    fig.show()
    display(df_pl[["label","spend","impressions","clicks","ctr","cpm"]]
            .sort_values("spend", ascending=False)
            .reset_index(drop=True))

In [ ]:
# ── Cell 9: Age & gender breakdown ───────────────────────────────────────────
demo = get(f"campaigns/{CAMPAIGN_ID}/demographics")
df_demo = pd.DataFrame(demo.get("demographics", []))

if not df_demo.empty:
    fig = px.bar(
        df_demo, x="age", y="spend", color="gender",
        barmode="group",
        title="Spend by Age & Gender",
        labels={"age": "Age Group", "spend": "Spend ($)"},
    )
    fig.show()

    if "leads" in df_demo.columns:
        fig2 = px.bar(
            df_demo, x="age", y="leads", color="gender",
            barmode="group",
            title="Leads by Age & Gender",
        )
        fig2.show()

In [ ]:
# ── Cell 10: Hourly performance heatmap ──────────────────────────────────────
hourly = get(f"campaigns/{CAMPAIGN_ID}/hourly")
df_h = pd.DataFrame(hourly.get("hourly", []))

if not df_h.empty and "hour" in df_h.columns:
    df_h = df_h.sort_values("hour")
    fig = go.Figure(go.Bar(
        x=df_h["hour"], y=df_h["spend"],
        marker_color=df_h["spend"],
        marker_colorscale="Blues",
    ))
    fig.update_layout(
        title="Hourly Spend Distribution",
        xaxis_title="Hour of Day (0–23)",
        yaxis_title="Spend ($)",
    )
    fig.show()

In [ ]:
# ── Cell 11: Geo & device breakdown ──────────────────────────────────────────
geo = get(f"campaigns/{CAMPAIGN_ID}/geo-device", date_from=DATE_FROM, date_to=DATE_TO)

devices = geo.get("devices", [])
countries = geo.get("countries", [])

if devices:
    df_dev = pd.DataFrame(devices)
    fig = px.pie(df_dev, names="device", values="spend", title="Spend by Device")
    fig.show()

if countries:
    df_geo = pd.DataFrame(countries)
    fig2 = px.choropleth(
        df_geo, locations="country", locationmode="country names",
        color="spend", title="Spend by Country",
        color_continuous_scale="Blues",
    )
    fig2.show()

In [ ]:
# ── Cell 12: Creative assets — top performers ─────────────────────────────────
cr = get(f"campaigns/{CAMPAIGN_ID}/creative-assets", sort_by="cpl")
df_cr = pd.DataFrame(cr.get("ads", []))

if not df_cr.empty:
    cols = [c for c in ["ad_name","spend","leads","cpl","ctr","impressions"] if c in df_cr.columns]
    display(df_cr[cols].sort_values("cpl").head(20)
            .reset_index(drop=True)
            .style.format({"spend": "${:.2f}", "cpl": "${:.2f}", "ctr": "{:.2f}%"})
            .background_gradient(subset=["cpl"], cmap="RdYlGn_r"))

---
## Competitor research (Meta Ad Library)

In [ ]:
# ── Cell 13: Search Ad Library ───────────────────────────────────────────────
SEARCH_QUERY = "Tokopedia"  # ← change to competitor brand name
SEARCH_COUNTRY = "ID"       # ISO country code

lib = get("ads-library/search", q=SEARCH_QUERY, country=SEARCH_COUNTRY, limit=20)
df_lib = pd.DataFrame(lib.get("ads", []))
print(f"{lib['count']} ads found for '{SEARCH_QUERY}' in {SEARCH_COUNTRY}")

if not df_lib.empty:
    show_cols = [c for c in ["page_name","ad_creative_bodies","ad_delivery_start_time","ad_delivery_stop_time"] if c in df_lib.columns]
    display(df_lib[show_cols].head(20))

---
## Sync

In [ ]:
# ── Cell 14: Sync status ──────────────────────────────────────────────────────
status = get("meta/status")
print(f"Last sync:    {status.get('last_sync_at', 'never')}")
print(f"Is syncing:   {status.get('is_syncing', False)}")
print(f"Campaigns:    {status.get('campaign_count', 0)}")

In [ ]:
# ── Cell 15: Trigger manual sync (run when you want fresh data) ───────────────
# Uncomment to run:
# r = requests.post(f"{BASE_URL}/meta/sync", headers=HEADERS)
# r.raise_for_status()
# print(r.json())